# 4 · Modern Portfolio Theory, optimisation & risk
$R_p=\sum_i w_iR_i \qquad \sigma_p=\sqrt{w^T\Sigma w} \qquad \text{Sharpe}=\dfrac{R_p-R_f}{\sigma_p}$

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import config
from src import data_loader as dl
OFFLINE = False   # True = synthetic sample data (no internet needed)

In [ ]:
if OFFLINE:
    data = dl.generate_sample_data(config.TICKERS, config.BENCHMARK)
    rf = config.DEFAULT_RISK_FREE
else:
    data = dl.fetch_prices(config.TICKERS + [config.BENCHMARK], config.START_DATE, config.END_DATE)
    rf = dl.fetch_risk_free(config.START_DATE, config.END_DATE)
prices = dl.close_matrix(data, config.TICKERS)
print(f'risk-free rate: {rf:.2%}'); prices.tail()

In [ ]:
from src.optimizer import optimise
opt = optimise(prices, rf, max_weight=config.MAX_WEIGHT, n_random=10_000)
(opt['weights'] * 100).round(1)  # weights in %

In [ ]:
pd.DataFrame(opt['performance']).T

In [ ]:
rnd, fr = opt['random'], opt['frontier']
plt.figure(figsize=(10, 6))
plt.scatter(rnd['Volatility'], rnd['Return'], c=rnd['Sharpe'], cmap='Blues', s=4); plt.colorbar(label='Sharpe')
plt.plot(fr['Volatility'], fr['Return'], 'k-', lw=2, label='Efficient frontier')
for name, m in (('Max Sharpe', '*'), ('Min Variance', 'D')):
    p = opt['performance'][name]; plt.scatter(p['Volatility'], p['Return'], marker=m, s=250, label=name)
plt.xlabel('Volatility'); plt.ylabel('Expected return'); plt.legend()

In [ ]:
from src.risk import risk_report
port = (opt['returns'] * opt['weights']['Max Sharpe']).sum(axis=1)
mkt = data[config.BENCHMARK]['Close'].pct_change().reindex(port.index)
pd.Series(risk_report(port, mkt, rf))

In [ ]:
from src.gain_estimator import estimate_gain
from src.forecast import run_forecast
w = opt['weights']['Max Sharpe']
fc = {t: run_forecast(data[t], t, data[config.BENCHMARK]).next_day for t in config.TICKERS}
pred = sum(w[t] * fc[t]['pred_return'] for t in config.TICKERS)
sd = sum((w[t] * fc[t]['return_sd'])**2 for t in config.TICKERS) ** 0.5
estimate_gain([10_000, 50_000, 100_000], pred, sd, risk_report(port)['VaR 95% (1d)'])